In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import normalize
from collections import defaultdict

H5_PATH = "../embeddings/cct_dinov2l_embeddings_v2.h5"
N_BLOCKS = 5          # blocks PER GROUP (5 city + 5 countryside = 10 total)
K_FACTOR = 0.5
INIT_TRAIN_RATIO = 0.8

print("Configuration set.")
print(f"  H5 path        : {H5_PATH}")
print(f"  Blocks per group: {N_BLOCKS}")
print(f"  k_factor       : {K_FACTOR}")
print(f"  Init train frac: {INIT_TRAIN_RATIO}")

In [ ]:
# ── 1. Load & clean ────────────────────────────────────────────────────────────
with h5py.File(H5_PATH, "r") as hf:
    raw_embeddings  = hf["embeddings"][:]
    raw_species     = np.array([s.decode() for s in hf["species"][:]])
    raw_strings     = [t.decode() for t in hf["date_captured"][:]]
    raw_locations   = np.array([l.decode() for l in hf["location"][:]])

temp_times = pd.to_datetime(raw_strings, errors="coerce")
mask       = temp_times.notna()

embeddings = normalize(raw_embeddings[mask], norm="l2")
species    = raw_species[mask]
timestamps = temp_times[mask]
locations  = raw_locations[mask]

print(f"Loaded {len(embeddings):,} embeddings after cleaning.")
print(f"Dropped {(~mask).sum()} rows with invalid timestamps.")

In [ ]:
# ── 2. Location → group assignment (majority-vote on rough threshold) ──────────
df = pd.DataFrame({"location": locations, "timestamp": timestamps})

rough_threshold = pd.Timestamp("2013-07-01")

location_counts = (
    df.groupby("location")
    .apply(lambda x: pd.Series({
        "count_early": (x["timestamp"] < rough_threshold).sum(),
        "count_late":  (x["timestamp"] >= rough_threshold).sum(),
    }))
    .reset_index()
)
location_counts["assigned_group"] = np.where(
    location_counts["count_early"] >= location_counts["count_late"],
    "group1", "group2"
)

group1_locs = set(location_counts[location_counts["assigned_group"] == "group1"]["location"])
group2_locs = set(location_counts[location_counts["assigned_group"] == "group2"]["location"])
assert len(group1_locs & group2_locs) == 0, "Location overlap between groups!"

borderline = location_counts[
    (location_counts["count_early"] > 0) & (location_counts["count_late"] > 0)
]
print(f"\nBorderline locations (data on both sides of threshold): {len(borderline)}")
print(borderline[["location", "count_early", "count_late", "assigned_group"]].to_string())
print(f"\nGroup 1 locations: {len(group1_locs)}")
print(f"Group 2 locations: {len(group2_locs)}")

In [ ]:
# ── 3. Sort ALL data chronologically, then derive per-group index arrays ───────
sort_idx   = np.argsort(timestamps)
embeddings = embeddings[sort_idx]
species    = species[sort_idx]
timestamps = timestamps[sort_idx]
locations  = locations[sort_idx]

dates_arr  = np.array([d.date() for d in timestamps])

# Boolean masks in the sorted array
g1_mask = np.isin(locations, list(group1_locs))
g2_mask = np.isin(locations, list(group2_locs))

# Absolute indices (into the sorted arrays) for each group
g1_indices = np.where(g1_mask)[0]
g2_indices = np.where(g2_mask)[0]

print(f"\nData sorted chronologically: {timestamps.min()} → {timestamps.max()}")
print(f"  Group 1 samples : {len(g1_indices):,}")
print(f"  Group 2 samples : {len(g2_indices):,}")
print(f"  Total           : {len(embeddings):,}")

In [ ]:
# ── 4. Date-aware block partitioning ──────────────────────────────────────────
def make_date_aware_blocks(dates_arr, global_indices, n_blocks):
    """
    Partition `global_indices` into `n_blocks` roughly-equal temporal blocks
    so that no calendar date is split across two blocks.

    Parameters
    ----------
    dates_arr     : np.ndarray of datetime.date, shape (N_total,)
                    Date for every sample in the globally-sorted array.
    global_indices: np.ndarray of int
                    Absolute positions (into dates_arr / embeddings / …) that
                    belong to this group, already in ascending time order.
    n_blocks      : int

    Returns
    -------
    List of np.ndarray – each array contains absolute indices for one block.
    """
    N = len(global_indices)
    target_block_size = N / n_blocks

    # Map each date to the global indices that fall on it (within this group)
    date_to_indices = defaultdict(list)
    for gi in global_indices:
        date_to_indices[dates_arr[gi]].append(gi)

    unique_dates = sorted(date_to_indices.keys())
    blocks    = [[] for _ in range(n_blocks)]
    cur_block = 0
    cumulative = 0

    for d in unique_dates:
        idxs = date_to_indices[d]
        blocks[cur_block].extend(idxs)
        cumulative += len(idxs)
        if cur_block < n_blocks - 1 and cumulative >= (cur_block + 1) * target_block_size:
            cur_block += 1

    return [np.array(b) for b in blocks if len(b) > 0]


g1_blocks = make_date_aware_blocks(dates_arr, g1_indices, N_BLOCKS)
g2_blocks = make_date_aware_blocks(dates_arr, g2_indices, N_BLOCKS)

In [ ]:
# ── 5. Summary ────────────────────────────────────────────────────────────────
def print_blocks(label, blocks, dates_arr):
    print(f"\n{'─'*60}")
    print(f"  {label}  ({len(blocks)} blocks)")
    print(f"  {'Block':>6}  {'Size':>7}  {'Date range'}")
    print(f"  {'─'*52}")
    for i, b in enumerate(blocks):
        d_min = dates_arr[b].min()
        d_max = dates_arr[b].max()
        print(f"    {i+1:>4}  {len(b):>7,}  {d_min}  →  {d_max}")

print_blocks("Group 1 (city)", g1_blocks, dates_arr)
print_blocks("Group 2 (countryside)", g2_blocks, dates_arr)

total = sum(len(b) for b in g1_blocks) + sum(len(b) for b in g2_blocks)
print(f"\nTotal samples across all blocks: {total:,}  (expected {len(embeddings):,})")
assert total == len(embeddings), "Sample count mismatch – some samples lost!"

In [ ]:
import numpy as np  # Imports NumPy for arrays, vector math, distances, and statistics.
from collections import defaultdict  # Imports defaultdict so each class can automatically get an empty gallery list.
from sklearn.decomposition import PCA  # Imports PCA for dimensionality reduction before classification.


class AdaptiveMahalanobisNCMClassifier:  # Defines a classifier class that combines NCM, PCA, open-set gating, and Mahalanobis-based updates.
    """
    Streaming cosine-NCM classifier with:  # High-level description of what the class implements.

    - L2-normalized embeddings  # Input embeddings are normalized to unit length.
    - Fixed PCA projection  # PCA is fit once and then reused for all future samples.
    - Per-class prototype  # Each species/class is represented by a mean embedding.
    - Diagonal Mahalanobis novelty gating  # Update decisions use a diagonal-covariance Mahalanobis score.
    - Duplicate rejection  # Very similar samples can be rejected as redundant.
    - Open-set recognition  # The classifier can flag samples as unknown.

    Strategy:  # Introduces the prediction and update logic at a high level.
    ----------
    Prediction:  # Explains how labels are predicted.
        cosine nearest class mean  # Prediction uses cosine-style distance to class prototypes.

    Update gating:  # Explains how new samples are accepted or rejected for updates.
        duplicate rejection +  # First remove near-duplicates.
        Mahalanobis middle-band acceptance  # Then accept only samples with a moderate novelty score.

    Notes:  # Additional design assumptions and limits.
    ------
    - PCA is fit ONCE during initial fit() and frozen forever.  # The feature space does not adapt after initialization.
    - Mahalanobis uses diagonal covariance per class.  # Feature correlations are ignored; only per-dimension variance is used.
    - Gallery buffer is stored fully (no memory cap).  # All accepted samples are kept, which may grow memory usage.
    """

    def __init__(  # Constructor that initializes hyperparameters and internal state.
        self,  # Refers to the current classifier instance.
        pca_dim=32,  # Number of PCA dimensions to keep.
        duplicate_thresh=0.97,  # Cosine similarity threshold above which a sample is treated as a duplicate.
        mahal_low=2.0,  # Lower Mahalanobis threshold; below this, a sample is too familiar.
        mahal_high=25.0,  # Upper Mahalanobis threshold; above this, a sample is too extreme/outlier-like.
        open_k_factor=2.0,  # Scales class-specific open-set radius using class spread.
        min_variance=1e-6  # Small floor to avoid divide-by-zero in variance-based calculations.
    ):

        # ---------- Hyperparameters ----------  # Marks the section storing tunable parameters.

        self.pca_dim = pca_dim  # Saves the requested PCA output dimension.

        # Reject embedding if too similar to gallery  # Explains the role of the next threshold.
        self.duplicate_thresh = duplicate_thresh  # Stores the duplicate rejection threshold.

        # Accept only if:  # Introduces the Mahalanobis acceptance rule.
        # mahal_low < M(x) < mahal_high  # A sample must be neither too typical nor too extreme.
        self.mahal_low = mahal_low  # Stores the lower Mahalanobis cutoff.
        self.mahal_high = mahal_high  # Stores the upper Mahalanobis cutoff.

        # Open-set radius scaling  # Explains the purpose of the next parameter.
        self.open_k_factor = open_k_factor  # Stores the multiplier used in open-set radius calculation.

        # Numerical stability  # Introduces the minimum allowed variance.
        self.min_variance = min_variance  # Stores a floor for per-dimension variance values.

        # ---------- PCA ----------  # Marks the PCA-related state section.

        self.pca = None  # Placeholder for the PCA model before fitting.
        self.pca_fitted = False  # Tracks whether PCA has already been trained.

        # ---------- Per-class state ----------  # Marks the state stored separately for each class.

        self.prototypes = {}  # Maps each class to its current prototype vector.
        self.counts = {}  # Maps each class to the number of accepted gallery samples.

        # diagonal variance per class  # Explains the next dictionary.
        self.vars = {}  # Maps each class to its per-dimension variance vector.

        # open-set stats  # Introduces statistics used for open-set radius computation.
        self.mean_dists = {}  # Stores mean prototype distance for each class.
        self.m2_dists = {}  # Stores the sum of squared deviations used to estimate class spread.

        # full gallery buffers  # Introduces the raw stored transformed samples for each class.
        self.galleries = defaultdict(list)  # Each class automatically starts with an empty sample list.

    # =========================================================  # Visual separator for readability.
    # Utilities  # Section title for helper methods.
    # =========================================================  # Visual separator for readability.

    def _normalize(self, x):  # Private helper that L2-normalizes a vector.

        x = np.asarray(x, dtype=np.float32)  # Converts the input to a NumPy float32 array.

        norm = np.linalg.norm(x)  # Computes the Euclidean length of the vector.

        if norm <= 0:  # Checks for zero-length or invalid norm cases.
            return x  # Returns the vector unchanged if it cannot be normalized safely.

        return x / norm  # Returns the unit-length version of the vector.

    def _transform(self, x):  # Private helper that applies the full feature transformation pipeline.
        """
        Normalize -> PCA -> Normalize  # Documents the exact transform order.
        """

        x = self._normalize(x)  # First normalize the raw embedding to unit length.

        x = self.pca.transform(x.reshape(1, -1))[0]  # Projects the vector into the frozen PCA space.

        x = self._normalize(x)  # Normalizes again so cosine comparisons in PCA space remain stable.

        return x.astype(np.float32)  # Returns the transformed embedding as float32.

    def _compute_open_radius(self, cls):  # Computes the class-specific radius used for open-set filtering.

        n = self.counts[cls]  # Reads how many samples currently belong to this class.

        if n < 2:  # If the class has too little data, spread estimation is unreliable.
            return 0.15  # Uses a fixed default radius for tiny classes.

        variance = max(  # Starts computing the variance of prototype distances for this class.
            0.0,  # Prevents negative variance due to numeric noise.
            self.m2_dists[cls] / (n - 1)  # Converts accumulated squared deviations into sample variance.
        )

        std = np.sqrt(variance)  # Converts variance to standard deviation.

        radius = (  # Builds the final open-set acceptance radius.
            self.mean_dists[cls]  # Starts from the average class distance.
            + max(0.05, self.open_k_factor * std)  # Adds a spread-dependent margin with a minimum floor.
        )

        return radius  # Returns the class-specific distance threshold.

    def _recompute_class_stats(self, cls):  # Rebuilds all statistics for one class from its full gallery.
        """
        Recompute prototype + diagonal variance  # Explains that this refreshes the class summary.
        from full gallery buffer.  # Notes that all stored samples are used.
        """

        X = np.asarray(self.galleries[cls])  # Converts the class gallery list into a matrix.

        self.counts[cls] = len(X)  # Updates the number of stored samples for the class.

        proto = X.mean(axis=0)  # Computes the arithmetic mean embedding for the class.

        proto = self._normalize(proto)  # Normalizes the prototype so cosine comparisons stay consistent.

        self.prototypes[cls] = proto  # Saves the updated class prototype.

        # diagonal covariance  # Introduces the variance estimate section.
        if len(X) < 2:  # If there is only one sample, variance cannot be estimated from data.

            var = np.ones(X.shape[1]) * self.min_variance  # Uses a tiny constant variance in every dimension.

        else:  # If there are at least two samples, compute data-driven variance.

            var = X.var(axis=0)  # Computes per-dimension variance across the class gallery.

            var = np.maximum(var, self.min_variance)  # Floors each variance term for numerical stability.

        self.vars[cls] = var  # Stores the class’s diagonal variance vector.

        # ----- open-set radius stats -----  # Introduces computation of prototype-distance statistics.

        dots = np.clip(X @ proto, -1.0, 1.0)  # Computes cosine-like dot products and clips them safely.

        dists = np.sqrt(  # Converts similarity scores into cosine-distance-style values.
            np.maximum(0.0, 2.0 - 2.0 * dots)  # Uses sqrt(2 - 2*cos(theta)) with a floor at zero.
        )

        mean_dist = dists.mean()  # Computes the average distance from gallery samples to the prototype.

        self.mean_dists[cls] = mean_dist  # Stores the mean class distance for later radius calculation.

        if len(dists) < 2:  # If only one distance exists, spread is undefined.

            self.m2_dists[cls] = 0.0  # Stores zero spread for a single-sample class.

        else:  # If multiple samples exist, estimate spread around the mean distance.

            self.m2_dists[cls] = np.sum(  # Accumulates squared deviations from the mean distance.
                (dists - mean_dist) ** 2  # This is the numerator used later for sample variance.
            )

    # =========================================================  # Visual separator for readability.
    # Initial fit  # Section title for the training/setup phase.
    # =========================================================  # Visual separator for readability.

    def fit(self, embeddings, labels):  # Fits the initial classifier state from a labeled seed set.
        """
        Heavy initial fit.  # Notes that this is the expensive initialization step.

        Steps:  # Introduces the fit pipeline.
        -------
        1. Normalize raw embeddings  # Standardize input embedding lengths.
        2. Fit PCA once  # Learn the projection basis from the initial data.
        3. Transform embeddings  # Move all samples into the PCA-normalized space.
        4. Build per-class galleries/statistics  # Initialize gallery buffers and class summaries.
        """

        embeddings = np.asarray(embeddings)  # Converts input embeddings into a NumPy array.
        labels = np.asarray(labels)  # Converts labels into a NumPy array for masking.

        # ---------- normalize raw embeddings ----------  # Introduces raw-space normalization.
        embeddings = np.asarray([  # Rebuilds the embedding matrix after normalizing each sample.
            self._normalize(x)  # Normalizes one embedding to unit length.
            for x in embeddings  # Iterates through all seed embeddings.
        ])

        # ---------- fit frozen PCA ----------  # Introduces PCA training on the normalized seed set.
        self.pca = PCA(  # Creates a PCA model object.
            n_components=self.pca_dim,  # Keeps only the configured number of principal components.
            random_state=42  # Fixes randomness for reproducibility.
        )

        self.pca.fit(embeddings)  # Learns the PCA projection from the normalized embeddings.

        self.pca_fitted = True  # Marks PCA as trained and ready for use.

        # ---------- transform embeddings ----------  # Introduces projection into PCA space.
        transformed = np.asarray([  # Builds a transformed version of every seed embedding.
            self._transform(x)  # Applies normalize -> PCA -> normalize to one sample.
            for x in embeddings  # Iterates through the normalized seed set.
        ])

        # ---------- build class galleries ----------  # Introduces per-class initialization.
        for cls in np.unique(labels):  # Loops once over each unique class label.

            X = transformed[labels == cls]  # Selects all transformed samples belonging to this class.

            self.galleries[cls] = [  # Stores this class’s transformed samples in its gallery buffer.
                x.copy() for x in X  # Copies each vector so later mutations do not affect the gallery.
            ]

            self._recompute_class_stats(cls)  # Recomputes prototype, variance, and radius stats for the class.

    # =========================================================  # Visual separator for readability.
    # Mahalanobis  # Section title for Mahalanobis distance logic.
    # =========================================================  # Visual separator for readability.

    def mahalanobis_distance(self, x, cls):  # Computes how unusual vector x is relative to one class.
        """
        Diagonal Mahalanobis distance.  # Explains that only per-dimension variance is used.
        """

        proto = self.prototypes[cls]  # Fetches the class prototype vector.
        var = self.vars[cls]  # Fetches the class’s per-dimension variance vector.

        delta = x - proto  # Computes feature-wise deviation from the prototype.

        m = np.sum(  # Starts the diagonal Mahalanobis calculation.
            (delta ** 2) / var  # For each feature: squared deviation divided by that feature’s variance.
        )

        return float(m)  # Returns the total Mahalanobis score as a Python float.

    # =========================================================  # Visual separator for readability.
    # Update gating  # Section title for deciding whether new samples should update the model.
    # =========================================================  # Visual separator for readability.

    def should_update(self, embedding, cls):  # Decides whether a new sample is useful enough to add.
        """
        Decide whether embedding adds useful information.  # Explains the purpose of this method.

        Returns:  # Documents the outputs.
        --------
        accept : bool  # Whether the sample should be added to the gallery.
        reason : str  # Why it was accepted or rejected.
        """

        if cls not in self.prototypes:  # If the class is completely new, it has no existing model state.
            return True, "new_class"  # Always accept the first example of a new class.

        x = self._transform(embedding)  # Transforms the incoming raw embedding into PCA-normalized space.

        gallery = np.asarray(self.galleries[cls])  # Loads the current gallery matrix for that class.

        # -------------------------------------------------  # Visual divider for duplicate rejection.
        # Duplicate rejection  # Section title for duplicate filtering.
        # -------------------------------------------------  # Visual divider for duplicate rejection.

        sims = gallery @ x  # Computes cosine-like similarities between the sample and all gallery items.

        max_sim = sims.max()  # Finds the most similar existing sample in the gallery.

        if max_sim >= self.duplicate_thresh:  # If the sample is too similar to something already stored,

            return False, "duplicate"  # reject it as redundant.

        # -------------------------------------------------  # Visual divider for Mahalanobis gating.
        # Mahalanobis novelty  # Section title for novelty/outlier filtering.
        # -------------------------------------------------  # Visual divider for Mahalanobis gating.

        m = self.mahalanobis_distance(x, cls)  # Computes how unusual the sample is for this class.

        # Too familiar -> redundant  # Explains the low-score case.
        if m < self.mahal_low:  # If the sample is very close to the class center relative to variance,

            return False, "low_novelty"  # reject it because it adds little new information.

        # Too extreme -> outlier  # Explains the high-score case.
        if m > self.mahal_high:  # If the sample is far beyond normal class variation,

            return False, "outlier"  # reject it as potentially noisy or mislabeled.

        # Sweet spot  # Explains the middle-band case.
        return True, "informative"  # Accept the sample because it is novel but still class-consistent.

    # =========================================================  # Visual separator for readability.
    # Streaming update  # Section title for online gallery growth.
    # =========================================================  # Visual separator for readability.

    def update(self, embedding, cls):  # Adds an accepted sample into the class model.
        """
        Add embedding to class gallery  # Explains the first update action.
        and recompute class statistics.  # Explains the second update action.
        """

        x = self._transform(embedding)  # Transforms the raw embedding into the model’s working feature space.

        self.galleries[cls].append(x)  # Appends the transformed sample to the class gallery.

        self._recompute_class_stats(cls)  # Recomputes prototype, variance, and open-set stats from all gallery samples.

    # =========================================================  # Visual separator for readability.
    # Prediction  # Section title for inference.
    # =========================================================  # Visual separator for readability.

    def predict(self, embedding):  # Predicts the nearest class and whether the sample should be treated as unknown.
        """
        Returns:  # Documents the outputs of prediction.
        --------
        pred_class  # The nearest accepted class label.
        is_unknown  # Whether the sample falls outside all class radii.
        """

        if not self.prototypes:  # If no classes have been learned yet,
            return None, True  # return no label and mark the sample as unknown.

        x = self._transform(embedding)  # Transforms the input embedding into PCA-normalized space.

        classes = list(self.prototypes.keys())  # Creates a stable list of known class labels.

        protos = np.asarray([  # Builds a matrix of all class prototype vectors.
            self.prototypes[c]  # Takes one prototype at a time.
            for c in classes  # Iterates over all known classes.
        ])

        dots = np.clip(  # Computes similarity between the sample and each class prototype.
            protos @ x,  # Matrix-vector product gives one dot product per class.
            -1.0,  # Lower clip bound for numerical safety.
            1.0  # Upper clip bound for numerical safety.
        )

        dists = np.sqrt(  # Converts those similarities into cosine-style distances.
            np.maximum(0.0, 2.0 - 2.0 * dots)  # Uses the same distance formula as elsewhere in the class.
        )

        # ---------- open-set filtering ----------  # Introduces radius-based unknown detection.

        valid = []  # Will hold indices of classes whose radius includes the sample.

        for i, cls in enumerate(classes):  # Loops over every known class and its index.

            radius = self._compute_open_radius(cls)  # Computes this class’s allowed in-class distance threshold.

            if dists[i] <= radius:  # If the sample is close enough to this class,
                valid.append(i)  # keep that class as a valid known-class candidate.

        # Unknown sample  # Explains the case where no class accepted the sample.
        if not valid:  # If the sample lies outside every class radius,

            nearest = np.argmin(dists)  # still find the nearest class for reference.
            return classes[nearest], True  # Return that nearest class but flag it as unknown.

        # Known sample  # Explains the case where at least one class accepted the sample.
        best = min(  # Chooses the closest class among the valid candidates.
            valid,  # Candidate class indices that passed the radius test.
            key=lambda i: dists[i]  # Selects the one with the smallest distance.
        )

        return classes[best], False  # Returns the accepted nearest class and marks it as known.

    # =========================================================  # Visual separator for readability.
    # Convenience  # Section title for helper properties.
    # =========================================================  # Visual separator for readability.

    @property  # Makes the next method accessible like an attribute.
    def known_species(self):  # Convenience property exposing all currently known classes.

        return set(self.prototypes.keys())  # Returns the set of class labels currently stored in the model.


In [ ]:
def evaluate_block(clf, embs, lbls):
    """
    Evaluate the classifier on a block of (emb, label) pairs.
    IMPORTANT: call this BEFORE updating the model on the same block.
    known_at_eval is snapshotted at call time to reflect what the model
    knew going into the block, not what it learns from it.
    Returns
    -------
    dict with keys:
        accuracy             – closed-set accuracy on known-class samples
        per_species_accuracy – per-species accuracy, same exclusion logic
        discovery_rate       – fraction of truly-new-species samples flagged unknown
        false_rejection_rate – fraction of known-class samples incorrectly flagged unknown
        false_acceptance_rate – fraction of truly-new-species samples NOT flagged unknown
                                (i.e. confidently misclassified as a known species)
        n_new_species        – count of new-species samples in this block
        n_known_samples      – count of known-class samples in this block
        total                – total samples in block
    """
    known_at_eval = clf.known_species.copy()
    sp_correct  = {sp: 0 for sp in known_at_eval}
    sp_total    = {sp: 0 for sp in known_at_eval}
    correct = 0
    new_species_total = 0
    correctly_flagged_new = 0
    known_but_flagged_unk = 0
    falsely_accepted_new = 0  # new species confidently called a known one

    for emb, lbl in zip(embs, lbls):
        pred, is_unknown = clf.predict(emb)
        truly_new = lbl not in known_at_eval
        if truly_new:
            new_species_total += 1
            if is_unknown:
                correctly_flagged_new += 1
            else:
                falsely_accepted_new += 1  # the silent failure, now counted
        else:
            sp_total[lbl] += 1
            if is_unknown:
                known_but_flagged_unk += 1
            elif pred == lbl:
                correct += 1
                sp_correct[lbl] += 1

    n_known_samples = sum(sp_total.values())
    accuracy = correct / n_known_samples if n_known_samples > 0 else float("nan")
    per_species_acc = {
        sp: (sp_correct[sp] / sp_total[sp] if sp_total[sp] > 0 else float("nan"))
        for sp in known_at_eval
    }
    discovery_rate = (
        correctly_flagged_new / new_species_total
        if new_species_total > 0
        else float("nan")
    )
    false_rejection_rate = (
        known_but_flagged_unk / n_known_samples
        if n_known_samples > 0
        else 0.0
    )
    # Complement of discovery_rate, but worth tracking explicitly:
    # a high false_acceptance_rate means novel animals are poisoning
    # your known-species confidence scores without any warning signal.
    false_acceptance_rate = (
        falsely_accepted_new / new_species_total
        if new_species_total > 0
        else 0.0
    )

    return dict(
        accuracy=accuracy,
        per_species_accuracy=per_species_acc,
        discovery_rate=discovery_rate,
        false_rejection_rate=false_rejection_rate,
        false_acceptance_rate=false_acceptance_rate,
        n_falsely_accepted=falsely_accepted_new,
        n_new_species=new_species_total,
        n_known_samples=n_known_samples,
        total=len(lbls),
    )

In [ ]:
import numpy as np

# ══════════════════════════════════════════════════════════════════════
# 1. CORE HELPERS & INITIALIZATION
# ══════════════════════════════════════════════════════════════════════

def _blk(idx, embeddings, species):
    return embeddings[idx], species[idx]

def _fmt(v):
    return f"{v:.3f}" if not np.isnan(v) else "n/a"

def _maybe_update_block(clf, embs, lbls):
    """
    Mahalanobis-gated streaming update over one block.
    """
    for e, l in zip(embs, lbls):
        accept, _ = clf.should_update(e, l)
        if accept:
            clf.update(e, l)

def _get_seed_clf(group_blocks, embeddings, species):
    """
    Fits the model ONLY on the first 80% of Block 1.
    Returns:
        clf: Initial model.
        hold_idx: The 20% of Block 1 that is 'unseen' for the first evaluation.
    """
    b0_idx = group_blocks[0]
    n_init = max(1, int(INIT_TRAIN_RATIO * len(b0_idx)))
    init_idx = b0_idx[:n_init]
    hold_idx = b0_idx[n_init:]

    clf = AdaptiveMahalanobisNCMClassifier(
        duplicate_thresh=0.97,
        mahal_low=2.0,
        mahal_high=50.0,
        open_k_factor=1.0,
        min_variance=1e-6,
    )
    clf.fit(embeddings[init_idx], species[init_idx])
    return clf, hold_idx

# ══════════════════════════════════════════════════════════════════════
# 2. INTERNAL PIPELINE (5-BLOCK TEMPORAL)
# ══════════════════════════════════════════════════════════════════════

def run_internal_pipeline(group_blocks, embeddings, species, label="Group", update_model=True):
    """
    Strictly follows: Evaluate on unseen -> Update.
    Block 1 baseline is measured on the 20% hold-out.
    """
    clf, b1_hold_idx = _get_seed_clf(group_blocks, embeddings, species)

    print("=" * 65)
    print(f"{label} INTERNAL — {'ADAPTIVE' if update_model else 'STATIC'}")
    print("=" * 65)

    results = []

    # --- Block 1: Evaluate 20% hold-out, then absorb if adaptive ---
    embs_h, lbls_h = _blk(b1_hold_idx, embeddings, species)
    n_before = len(clf.known_species)
    metrics = evaluate_block(clf, embs_h, lbls_h)

    if update_model:
        _maybe_update_block(clf, embs_h, lbls_h)

    n_after = len(clf.known_species)
    results.append({
        "block": 1,
        **metrics,
        "n_known_before": n_before,
        "n_known_after": n_after
    })

    print(
        f"  Block 1 | Acc: {_fmt(metrics['accuracy'])} | "
        f"DR: {_fmt(metrics['discovery_rate'])} | Known: {n_before} → {n_after}"
    )

    # --- Blocks 2-5: Evaluate whole block, then absorb if adaptive ---
    for i, blk_idx in enumerate(group_blocks[1:], start=2):
        embs, lbls = _blk(blk_idx, embeddings, species)
        n_before = len(clf.known_species)
        metrics = evaluate_block(clf, embs, lbls)

        if update_model:
            _maybe_update_block(clf, embs, lbls)

        n_after = len(clf.known_species)
        results.append({
            "block": i,
            **metrics,
            "n_known_before": n_before,
            "n_known_after": n_after
        })

        print(
            f"  Block {i} | Acc: {_fmt(metrics['accuracy'])} | "
            f"DR: {_fmt(metrics['discovery_rate'])} | Known: {n_before} → {n_after}"
        )

    print("-" * 65)
    return results

# ══════════════════════════════════════════════════════════════════════
# 3. TRANSFER MATRIX (CROSS-TESTING)
# ══════════════════════════════════════════════════════════════════════

def get_scenario_trajectories(matrix_ad, matrix_st):
    """
    Extracts the Baseline (Static model, seed only) and
    the Method (Adaptive model, after all 5 source blocks).
    """
    baseline = [m["accuracy"] for m in matrix_st[0]]
    method = [m["accuracy"] for m in matrix_ad[-1]]
    return baseline, method

def run_transfer_matrix(source_blocks, target_blocks, embeddings, species,
                        source_label="Source", target_label="Target", update_model=True):
    """
    Source-domain model is seeded on 80% of source Block 1.
    Then the source Block 1 hold-out is absorbed to complete Block 1 knowledge.
    Later source blocks are absorbed only in the adaptive setting.
    """
    clf, b1_hold_idx = _get_seed_clf(source_blocks, embeddings, species)

    # Complete source Block 1 knowledge using Mahalanobis-gated updates
    embs_h, lbls_h = _blk(b1_hold_idx, embeddings, species)
    _maybe_update_block(clf, embs_h, lbls_h)

    print("=" * 75)
    print(f"TRANSFER MATRIX: {source_label} -> {target_label} ({'Adaptive' if update_model else 'Static'})")
    print("=" * 75)
    header = "             " + "".join(f"  {target_label[:1]}[{j+1}]  " for j in range(len(target_blocks)))
    print(header)

    matrix = []
    for i, src_idx in enumerate(source_blocks, start=1):
        # For adaptive transfer, absorb source blocks 2..N before evaluating this row
        if update_model and i > 1:
            embs_s, lbls_s = _blk(src_idx, embeddings, species)
            _maybe_update_block(clf, embs_s, lbls_s)

        row_metrics = []
        row_str = f"  M_{source_label[:1]}[{i}] k={len(clf.known_species):>2} |"

        for tgt_idx in target_blocks:
            embs_t, lbls_t = _blk(tgt_idx, embeddings, species)
            m = evaluate_block(clf, embs_t, lbls_t)
            row_metrics.append(m)
            row_str += f"  {_fmt(m['accuracy'])} "

        matrix.append(row_metrics)
        print(row_str)

    print("-" * 75)
    return matrix

# ══════════════════════════════════════════════════════════════════════
# 5. EXECUTION
# ══════════════════════════════════════════════════════════════════════

# -- Internal Pipelines --
city_ad = run_internal_pipeline(g1_blocks, embeddings, species, "City", update_model=True)
city_st = run_internal_pipeline(g1_blocks, embeddings, species, "City", update_model=False)
rural_ad = run_internal_pipeline(g2_blocks, embeddings, species, "Rural", update_model=True)
rural_st = run_internal_pipeline(g2_blocks, embeddings, species, "Rural", update_model=False)

# -- Transfer Matrices --
#city_to_rural_ad = run_transfer_matrix(g1_blocks, g2_blocks, embeddings, species, "City", "Rural", True)
#rural_to_city_ad = run_transfer_matrix(g2_blocks, g1_blocks, embeddings, species, "Rural", "City", True)
#city_to_rural_st = run_transfer_matrix(g1_blocks, g2_blocks, embeddings, species, "City", "Rural", False)
#rural_to_city_st = run_transfer_matrix(g2_blocks, g1_blocks, embeddings, species, "Rural", "City", False)
